## Personal Parameters

- **SID4:** 1980
- **SEED:** 1980
- **SLICE:** 980
- **HP_ID:** 0
- **CLS_A:** 0
- **CLS_B:** 1



In [ ]:
# Step 1: Install required libraries for the RAG pipeline

# langchain: core RAG framework
# langchain-community: document loaders and vector stores
# langchain-huggingface: Hugging Face embedding integration
# sentence-transformers: generates vector embeddings
# faiss-cpu: stores and searches chunk embeddings
# wikipedia: retrieves Wikipedia page content

!pip install -q langchain langchain-community langchain-huggingface \
sentence-transformers faiss-cpu wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# Step 2: Import required libraries for the RAG pipeline

# WikipediaLoader:
# Loads Wikipedia pages as LangChain Document objects.
from langchain_community.document_loaders import WikipediaLoader

# RecursiveCharacterTextSplitter:
# Splits large Wikipedia documents into smaller text chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# HuggingFaceEmbeddings:
# Converts text chunks into numerical embedding vectors.
from langchain_huggingface import HuggingFaceEmbeddings

# FAISS:
# Vector store used to save embeddings and retrieve similar chunks.
from langchain_community.vectorstores import FAISS

# PromptTemplate:
# Used to explicitly construct the prompt sent to the LLM.
from langchain_core.prompts import PromptTemplate

# Basic utilities
import pandas as pd
import numpy as np

print("All required libraries imported successfully!")

/tmp/ipykernel_4045/3822773970.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


All required libraries imported successfully!


In [ ]:
# Step 3: Define the 10 movies for our RAG knowledge base

# These movie titles will be used to load their
# corresponding Wikipedia pages using LangChain.
movie_titles = [
    "The Shawshank Redemption",
    "The Godfather",
    "The Dark Knight",
    "Inception",
    "Interstellar",
    "Titanic (1997 film)",
    "The Matrix",
    "Forrest Gump",
    "Gladiator (2000 film)",
    "Jurassic Park (film)"
]

# Display the selected movies.
print("Movies selected for the RAG dataset:")
print("------------------------------------")

for i, movie in enumerate(movie_titles, start=1):
    print(f"{i}. {movie}")

# Verify that exactly 10 movies were selected.
print("\nTotal number of movies:", len(movie_titles))

Movies selected for the RAG dataset:
------------------------------------
1. The Shawshank Redemption
2. The Godfather
3. The Dark Knight
4. Inception
5. Interstellar
6. Titanic (1997 film)
7. The Matrix
8. Forrest Gump
9. Gladiator (2000 film)
10. Jurassic Park (film)

Total number of movies: 10


In [ ]:
# Step 4: Load the 10 movie Wikipedia pages using LangChain

# Create an empty list to store the loaded documents.
movie_documents = []

# Loop through each of the 10 movie titles.
for movie in movie_titles:

    print(f"Loading: {movie}")

    try:
        # Create a WikipediaLoader for the current movie.
        # load_max_docs=1 ensures that only the most relevant
        # Wikipedia page is returned for each movie.
        loader = WikipediaLoader(
            query=movie,
            load_max_docs=1,
            doc_content_chars_max=100000
        )

        # Load the Wikipedia page as a LangChain Document.
        docs = loader.load()

        # Check whether a document was successfully retrieved.
        if len(docs) > 0:

            # Store the movie name in metadata so that later
            # we can identify which movie a retrieved chunk came from.
            docs[0].metadata["movie"] = movie

            # Add the document to our collection.
            movie_documents.append(docs[0])

            print(f"  Successfully loaded: {docs[0].metadata.get('title')}")

        else:
            print("  No document found.")

    except Exception as e:
        print(f"  Error loading {movie}: {e}")


# --------------------------------------------------
# Verify the final dataset
# --------------------------------------------------

print("\n------------------------------------")
print("Wikipedia loading completed.")
print("Total documents loaded:", len(movie_documents))

# Display the titles of all loaded documents.
print("\nLoaded Wikipedia documents:")

for i, doc in enumerate(movie_documents, start=1):
    print(f"{i}. {doc.metadata.get('title')}")

Loading: The Shawshank Redemption
  Successfully loaded: The Shawshank Redemption
Loading: The Godfather
  Successfully loaded: The Godfather
Loading: The Dark Knight
  Successfully loaded: The Dark Knight
Loading: Inception
  Successfully loaded: Inception
Loading: Interstellar
  Successfully loaded: Interstellar (film)
Loading: Titanic (1997 film)
  Successfully loaded: Titanic (1997 film)
Loading: The Matrix
  Successfully loaded: The Matrix
Loading: Forrest Gump
  Successfully loaded: Forrest Gump
Loading: Gladiator (2000 film)
  Successfully loaded: Gladiator (2000 film)
Loading: Jurassic Park (film)
  Successfully loaded: Jurassic Park (franchise)

------------------------------------
Wikipedia loading completed.
Total documents loaded: 10

Loaded Wikipedia documents:
1. The Shawshank Redemption
2. The Godfather
3. The Dark Knight
4. Inception
5. Interstellar (film)
6. Titanic (1997 film)
7. The Matrix
8. Forrest Gump
9. Gladiator (2000 film)
10. Jurassic Park (franchise)


In [ ]:
# Step 5: Split the Wikipedia documents into smaller text chunks

# Create the text splitter using the assignment-required settings.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Maximum size of each text chunk
    chunk_overlap=50     # 50 characters overlap between consecutive chunks
)

# Split all 10 Wikipedia documents.
movie_chunks = text_splitter.split_documents(movie_documents)

# --------------------------------------------------
# Verify the chunking process
# --------------------------------------------------

print("Document chunking completed successfully!")
print("Original documents:", len(movie_documents))
print("Total chunks created:", len(movie_chunks))

# Display one example chunk.
print("\nExample Chunk:")
print("------------------------------------")
print(movie_chunks[0].page_content)

print("\nChunk metadata:")
print(movie_chunks[0].metadata)

Document chunking completed successfully!
Original documents: 10
Total chunks created: 1690

Example Chunk:
------------------------------------
The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Hayworth and Shawshank Redemption. The film tells the story of banker Andy Dufresne (Tim Robbins), who is sentenced to life in Shawshank State Penitentiary for the murders of his wife and her lover, despite his claims of innocence. Over the following two decades, he befriends a fellow prisoner, contraband smuggler Ellis "Red" Redding (Morgan Freeman), and

Chunk metadata:
{'title': 'The Shawshank Redemption', 'summary': 'The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Hayworth and Shawshank Redemption. The film tells the story of banker Andy Dufresne (Tim Robbins), who is sentenced to life in Shawshank State Penitenti

In [ ]:
# Step 6: Create embeddings for all Wikipedia movie chunks

# Load a pretrained sentence-transformer embedding model.
# This model converts text into dense numerical vectors.
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Generate embeddings and store them in a FAISS vector database.
vector_store = FAISS.from_documents(
    documents=movie_chunks,
    embedding=embedding_model
)

print("Embeddings created successfully!")
print("FAISS vector store created successfully!")
print("Total chunks indexed:", len(movie_chunks))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created successfully!
FAISS vector store created successfully!
Total chunks indexed: 1690


In [ ]:
# Step 7: Create the retriever

# Convert the FAISS vector store into a retriever.
# k=3 means the retriever will return the top 3
# most relevant chunks for each question.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")
print("Number of chunks retrieved per question: 3")

Retriever created successfully!
Number of chunks retrieved per question: 3


In [ ]:
# Step 8: Load a local instruction LLM and connect it to LangChain

import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# Check whether Colab GPU is available
device = 0 if torch.cuda.is_available() else -1

print("Using GPU" if device == 0 else "Using CPU")

# Create a text-generation pipeline.
# Qwen2.5-0.5B-Instruct is a small instruction-following model
# that can run on a Colab T4 GPU.
generation_pipeline = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=device,
    max_new_tokens=150,
    do_sample=False,
    return_full_text=False
)

# Wrap the Hugging Face pipeline as a LangChain LLM.
llm = HuggingFacePipeline(
    pipeline=generation_pipeline
)

print("LangChain LLM created successfully!")

Using GPU


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LangChain LLM created successfully!


In [ ]:
# Step 9: Create the prompt template for the RAG system

# PromptTemplate explicitly defines how the retrieved
# context and user question will be given to the LLM.

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are answering questions about movies using Wikipedia context.

Use ONLY the information provided in the context below.
If the answer is not available in the context, say:
"I could not find the answer in the retrieved context."

Context:
{context}

Question:
{question}

Answer:
"""
)

print("Prompt template created successfully!")

Prompt template created successfully!


In [ ]:
# Step 10: Define five answerable questions for the RAG experiment

rag_questions = [
    "Who directed The Shawshank Redemption?",
    "Who played Vito Corleone in The Godfather?",
    "Who directed Inception?",
    "Who composed the music for Interstellar?",
    "Who directed Jurassic Park?"
]

print("Five RAG questions:")
print("-------------------")

for i, question in enumerate(rag_questions, start=1):
    print(f"{i}. {question}")

Five RAG questions:
-------------------
1. Who directed The Shawshank Redemption?
2. Who played Vito Corleone in The Godfather?
3. Who directed Inception?
4. Who composed the music for Interstellar?
5. Who directed Jurassic Park?


In [ ]:
# Step 11: Retrieve the top-3 chunks for Question 1

# Select the first question.
question_1 = rag_questions[0]

print("Question 1:")
print(question_1)

# Retrieve the top-3 most relevant chunks.
retrieved_docs_q1 = retriever.invoke(question_1)

print("\nTop-3 Retrieved Chunks")
print("----------------------")

# Display each retrieved chunk along with its source movie.
for i, doc in enumerate(retrieved_docs_q1, start=1):

    print(f"\nChunk {i}")
    print("Movie:", doc.metadata.get("movie"))
    print("Wikipedia Title:", doc.metadata.get("title"))
    print("Content:")
    print(doc.page_content)

    print("\n" + "-" * 80)

Question 1:
Who directed The Shawshank Redemption?

Top-3 Retrieved Chunks
----------------------

Chunk 1
Movie: The Shawshank Redemption
Wikipedia Title: The Shawshank Redemption
Content:
Darabont purchased the film rights to King's story in 1987, but development did not begin until five years later, when he wrote the script over eight weeks. Two weeks after submitting his script to Castle Rock Entertainment, Darabont secured a $25 million budget to produce The Shawshank Redemption, which started pre-production in January 1993. While the film is set in Maine, principal photography took place from June to August 1993 almost entirely in Mansfield, Ohio, with the Ohio State

--------------------------------------------------------------------------------

Chunk 2
Movie: The Shawshank Redemption
Wikipedia Title: The Shawshank Redemption
Content:
The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Haywo

In [ ]:
# Step 12: Generate the final RAG answer for Question 1

# Combine the top-3 retrieved chunks into one context.
context_q1 = "\n\n".join(
    doc.page_content for doc in retrieved_docs_q1
)

# Insert the retrieved context and question
# into the prompt template created in Step 9.
formatted_prompt_q1 = rag_prompt.format(
    context=context_q1,
    question=question_1
)

# Send the completed prompt to the LLM.
final_answer_q1 = llm.invoke(formatted_prompt_q1)

# Display the question and final generated answer.
print("Question:")
print(question_1)

print("\nFinal RAG Answer:")
print("-----------------")
print(final_answer_q1)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question:
Who directed The Shawshank Redemption?

Final RAG Answer:
-----------------
Frank Darabont directed The Shawshank Redemption. 

I could not find the answer in the retrieved context.


In [ ]:
# Step 12B: Improve the RAG prompt to avoid contradictory answers

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Answer the question using ONLY the context below.

Rules:
1. If the answer is present in the context, give only the answer.
2. If the answer is NOT present in the context, say exactly:
   I could not find the answer in the retrieved context.
3. Do not give both an answer and the fallback sentence.

Context:
{context}

Question:
{question}

Answer:
"""
)

print("Improved RAG prompt created successfully!")

Improved RAG prompt created successfully!


In [ ]:
# Step 12: Generate the final RAG answer for Question 1

# Combine the top-3 retrieved chunks into one context.
context_q1 = "\n\n".join(
    doc.page_content for doc in retrieved_docs_q1
)

# Insert the retrieved context and question
# into the prompt template created in Step 9.
formatted_prompt_q1 = rag_prompt.format(
    context=context_q1,
    question=question_1
)

# Send the completed prompt to the LLM.
final_answer_q1 = llm.invoke(formatted_prompt_q1)

# Display the question and final generated answer.
print("Question:")
print(question_1)

print("\nFinal RAG Answer:")
print("-----------------")
print(final_answer_q1)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Who directed The Shawshank Redemption?

Final RAG Answer:
-----------------
Frank Darabont directed The Shawshank Redemption. According to the provided context, Frank Darabont wrote and directed the film, based on the 1982 Stephen King novella. He also wrote the screenplay over eight weeks and secured a $25 million budget to produce it. Therefore, the answer is Frank Darabont.


In [ ]:
# Step 13: Run RAG for Questions 2 to 5

def run_rag_question(question):

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    # Retrieve top-3 relevant chunks
    retrieved_docs = retriever.invoke(question)

    print("\nTOP-3 RETRIEVED CHUNKS")
    print("-" * 100)

    # Display each retrieved chunk
    for i, doc in enumerate(retrieved_docs, start=1):

        print(f"\nChunk {i}")
        print("Movie:", doc.metadata.get("movie"))
        print("Wikipedia Title:", doc.metadata.get("title"))
        print("Content:")
        print(doc.page_content)

        print("\n" + "-" * 100)

    # Combine retrieved chunks into a single context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Insert context and question into the prompt
    formatted_prompt = rag_prompt.format(
        context=context,
        question=question
    )

    # Generate the final answer using the LLM
    answer = llm.invoke(formatted_prompt)

    print("\nFINAL RAG ANSWER:")
    print(answer)

    return retrieved_docs, answer


# Run Questions 2 to 5
rag_results = {}

for i in range(1, 5):

    question = rag_questions[i]

    retrieved_docs, answer = run_rag_question(question)

    rag_results[i + 1] = {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "answer": answer
    }

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Who played Vito Corleone in The Godfather?

TOP-3 RETRIEVED CHUNKS
----------------------------------------------------------------------------------------------------

Chunk 1
Movie: The Godfather
Wikipedia Title: The Godfather
Content:
== Plot ==
In the mid-1940s, Michael Corleone is a World War II veteran and the youngest son of Don Vito Corleone. Vito is head of the Corleone mafia family and runs it with his eldest son Santino a.k.a. Sonny and adopted son Tom Hagen, a lawyer who is the family's consigliere. At his sister Connie's wedding to Carlo Rizzi, Michael introduces his family to his girlfriend, Kay Adams. Michael tells Kay stories of his family but makes it clear that he wants no part in his family's business.

----------------------------------------------------------------------------------------------------

Chunk 2
Movie: The Godfather
Wikipedia Title: The Godfather
Content:
under patriarch Vito Corleone (Brando) and the transformation of his youngest son, Mic

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL RAG ANSWER:
Michael Corleone (played by Pacino)
Fallback sentence: I could not find the answer in the retrieved context.

QUESTION:
Who directed Inception?

TOP-3 RETRIEVED CHUNKS
----------------------------------------------------------------------------------------------------

Chunk 1
Movie: Inception
Wikipedia Title: Inception
Content:
Inception is a 2010  science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating his targets' subconscious. He is offered a chance to have his criminal history erased as payment for implanting an idea into a target's subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page, Tom Hardy, Cillian

----------------------------------------------------------------------------------------------------

Chunk 2
Movie: Inception
Wikipedia Title: Incepti

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL RAG ANSWER:
Christopher Nolan directed Inception. According to the provided context, Christopher Nolan wrote and directed the film Inception, which was produced by him along with his wife Emma Thomas. No other director was mentioned in relation to this film. Therefore, the answer is: Christopher Nolan.

QUESTION:
Who composed the music for Interstellar?

TOP-3 RETRIEVED CHUNKS
----------------------------------------------------------------------------------------------------

Chunk 1
Movie: Interstellar
Wikipedia Title: Interstellar (film)
Content:
organ of the Temple Church, London, would be the primary instrument for the score. Zimmer conducted 45 scoring sessions for Interstellar, three times more than for Inception. The soundtrack was released on November 17, 2014.

----------------------------------------------------------------------------------------------------

Chunk 2
Movie: Interstellar
Wikipedia Title: Interstellar (film)
Content:
clumsy sound mixing", and "thin cha

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL RAG ANSWER:
The answer is: Zimmer
I could not find the answer in the retrieved context.

QUESTION:
Who directed Jurassic Park?

TOP-3 RETRIEVED CHUNKS
----------------------------------------------------------------------------------------------------

Chunk 1
Movie: Jurassic Park (film)
Wikipedia Title: Jurassic Park (franchise)
Content:
producer for each subsequent film, starting with Jurassic Park III (2001).

----------------------------------------------------------------------------------------------------

Chunk 2
Movie: Jurassic Park (film)
Wikipedia Title: Jurassic Park (franchise)
Content:
was one of the reasons I wanted to make Jurassic Park"."

----------------------------------------------------------------------------------------------------

Chunk 3
Movie: Jurassic Park (film)
Wikipedia Title: Jurassic Park (franchise)
Content:
The films begin as an adaptation of Michael Crichton's 1990 novel with the film Jurassic Park. After an incident with a velociraptor in Ju

In [ ]:
# Step 13B: Load the exact Jurassic Park movie Wikipedia page

import os
from langchain_community.document_loaders import WebBaseLoader

# Set a user agent for the web loader
os.environ["USER_AGENT"] = "Mozilla/5.0"

# Exact Wikipedia page for the 1993 Jurassic Park film
jurassic_url = "https://en.wikipedia.org/wiki/Jurassic_Park_(film)"

# Load the page directly using a LangChain Document Loader
jurassic_loader = WebBaseLoader(jurassic_url)
jurassic_docs = jurassic_loader.load()

# Add consistent metadata
jurassic_docs[0].metadata["movie"] = "Jurassic Park (film)"
jurassic_docs[0].metadata["title"] = "Jurassic Park"

# Add the correct document back to our dataset
movie_documents.append(jurassic_docs[0])

# Verify
print("Loaded title:", jurassic_docs[0].metadata.get("title"))
print("Total documents:", len(movie_documents))

Loaded title: Jurassic Park
Total documents: 10


In [ ]:
# Step 13C: Rebuild chunks and FAISS vector store after fixing Jurassic Park

# Split the corrected 10 documents again
movie_chunks = text_splitter.split_documents(movie_documents)

print("Total chunks after correction:", len(movie_chunks))

# Rebuild the FAISS vector store using the corrected chunks
vector_store = FAISS.from_documents(
    documents=movie_chunks,
    embedding=embedding_model
)

# Recreate the top-3 retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("FAISS vector store rebuilt successfully!")
print("Retriever recreated successfully!")

Total chunks after correction: 2044
FAISS vector store rebuilt successfully!
Retriever recreated successfully!


In [ ]:
# Step 13D: Rerun Question 5 after fixing Jurassic Park

question_5 = rag_questions[4]

print("Question 5:")
print(question_5)

# Retrieve top-3 chunks using the corrected vector store
retrieved_docs_q5 = retriever.invoke(question_5)

print("\nTop-3 Retrieved Chunks")
print("----------------------")

for i, doc in enumerate(retrieved_docs_q5, start=1):
    print(f"\nChunk {i}")
    print("Movie:", doc.metadata.get("movie"))
    print("Wikipedia Title:", doc.metadata.get("title"))
    print("Content:")
    print(doc.page_content)
    print("\n" + "-" * 80)

# Combine retrieved chunks
context_q5 = "\n\n".join(
    doc.page_content for doc in retrieved_docs_q5
)

# Build prompt
formatted_prompt_q5 = rag_prompt.format(
    context=context_q5,
    question=question_5
)

# Generate final answer
final_answer_q5 = llm.invoke(formatted_prompt_q5)

print("\nFinal RAG Answer:")
print("-----------------")
print(final_answer_q5)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 5:
Who directed Jurassic Park?

Top-3 Retrieved Chunks
----------------------

Chunk 1
Movie: Jurassic Park (film)
Wikipedia Title: Jurassic Park
Content:
For the film, Kiley was cast in the same role,[126] although James Earl Jones was also considered; Jones later hosted the documentary The Making of Jurassic Park.[127]

--------------------------------------------------------------------------------

Chunk 2
Movie: Jurassic Park (film)
Wikipedia Title: Jurassic Park
Content:
This article may be too long to read and navigate comfortably. Consider splitting content into sub-articles, condensing it, or adding subheadings. Please discuss this issue on the article's talk page. (July 2026)
Jurassic ParkTheatrical release poster by John AlvinDirected bySteven SpielbergScreenplay by
Michael Crichton
David Koepp
Based onJurassic Parkby Michael CrichtonProduced by
Kathleen Kennedy
Gerald R. Molen
Starring
Sam Neill
Laura Dern
Jeff Goldblum
Richard Attenborough
Bob Peck

--------------

In [ ]:
# Step 14: Create the manual retrieval evaluation table

evaluation_data = [
    {
        "Question": "Who directed The Shawshank Redemption?",
        "Correct Answer": "Frank Darabont",
        "Top-3 Contains Answer?": "Yes",
        "First Relevant Rank": 2,
        "Final RAG Answer Correct?": "Yes",
        "Failure Type": "None"
    },
    {
        "Question": "Who played Vito Corleone in The Godfather?",
        "Correct Answer": "Marlon Brando",
        "Top-3 Contains Answer?": "Yes",
        "First Relevant Rank": 2,
        "Final RAG Answer Correct?": "No",
        "Failure Type": "LLM generation failure"
    },
    {
        "Question": "Who directed Inception?",
        "Correct Answer": "Christopher Nolan",
        "Top-3 Contains Answer?": "Yes",
        "First Relevant Rank": 1,
        "Final RAG Answer Correct?": "Yes",
        "Failure Type": "None"
    },
    {
        "Question": "Who composed the music for Interstellar?",
        "Correct Answer": "Hans Zimmer",
        "Top-3 Contains Answer?": "Yes",
        "First Relevant Rank": 1,
        "Final RAG Answer Correct?": "Yes",
        "Failure Type": "None"
    },
    {
        "Question": "Who directed Jurassic Park?",
        "Correct Answer": "Steven Spielberg",
        "Top-3 Contains Answer?": "Yes",
        "First Relevant Rank": 2,
        "Final RAG Answer Correct?": "No",
        "Failure Type": "LLM generation failure"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

,Question,Correct Answer,Top-3 Contains Answer?,First Relevant Rank,Final RAG Answer Correct?,Failure Type
0,Who directed The Shawshank Redemption?,Frank Darabont,Yes,2,Yes,None
1,Who played Vito Corleone in The Godfather?,Marlon Brando,Yes,2,No,LLM generation failure
2,Who directed Inception?,Christopher Nolan,Yes,1,Yes,None
3,Who composed the music for Interstellar?,Hans Zimmer,Yes,1,Yes,None
4,Who directed Jurassic Park?,Steven Spielberg,Yes,2,No,LLM generation failure


In [ ]:
# Step 15: Calculate Retrieval Success Rate

# Count how many questions had the correct information
# somewhere in the top-3 retrieved chunks.
successful_retrievals = (
    evaluation_df["Top-3 Contains Answer?"] == "Yes"
).sum()

total_questions = len(evaluation_df)

# Calculate retrieval success rate
retrieval_success_rate = successful_retrievals / total_questions

print("Retrieval Evaluation")
print("--------------------")
print("Successful retrievals:", successful_retrievals)
print("Total questions:", total_questions)

print(
    "Retrieval Success Rate:",
    f"{successful_retrievals}/{total_questions} = "
    f"{retrieval_success_rate:.2%}"
)

Retrieval Evaluation
--------------------
Successful retrievals: 5
Total questions: 5
Retrieval Success Rate: 5/5 = 100.00%


In [ ]:
# Step 16: Analyze two RAG failures observed in the experiment

print("RAG Failure Analysis")
print("====================")

print("\nFailure 1: The Godfather")
print("------------------------")
print("Question: Who played Vito Corleone in The Godfather?")
print("Correct Answer: Marlon Brando")
print("Retrieval: Successful - relevant information appeared in the top-3 chunks.")
print("Generated Answer: Michael Corleone (played by Pacino)")
print(
    "Analysis: The retriever successfully returned information connecting "
    "Marlon Brando with Vito Corleone. However, the LLM confused the "
    "characters Vito Corleone and Michael Corleone and generated the wrong "
    "answer. Therefore, this was an LLM generation failure rather than a "
    "retrieval failure."
)

print("\nFailure 2: Jurassic Park")
print("------------------------")
print("Question: Who directed Jurassic Park?")
print("Correct Answer: Steven Spielberg")
print("Retrieval: Successful - Steven Spielberg appeared in retrieved chunk 2.")
print("Generated Answer: I could not find the answer in the retrieved context.")
print(
    "Analysis: The retrieved context explicitly contained "
    "'Directed by Steven Spielberg', but the LLM failed to extract this "
    "information and incorrectly returned the fallback response. Therefore, "
    "this was also an LLM generation failure rather than a retrieval failure."
)

RAG Failure Analysis

Failure 1: The Godfather
------------------------
Question: Who played Vito Corleone in The Godfather?
Correct Answer: Marlon Brando
Retrieval: Successful - relevant information appeared in the top-3 chunks.
Generated Answer: Michael Corleone (played by Pacino)
Analysis: The retriever successfully returned information connecting Marlon Brando with Vito Corleone. However, the LLM confused the characters Vito Corleone and Michael Corleone and generated the wrong answer. Therefore, this was an LLM generation failure rather than a retrieval failure.

Failure 2: Jurassic Park
------------------------
Question: Who directed Jurassic Park?
Correct Answer: Steven Spielberg
Retrieval: Successful - Steven Spielberg appeared in retrieved chunk 2.
Generated Answer: I could not find the answer in the retrieved context.
Analysis: The retrieved context explicitly contained 'Directed by Steven Spielberg', but the LLM failed to extract this information and incorrectly returned the

In [ ]:
# Step 17: Create alternative chunking configuration

# Original: chunk_size=500, chunk_overlap=50
# Alternative: chunk_size=1000, chunk_overlap=100

alt_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

# Split the same 10 Wikipedia documents
alt_movie_chunks = alt_text_splitter.split_documents(movie_documents)

print("Original number of chunks:", len(movie_chunks))
print("Alternative number of chunks:", len(alt_movie_chunks))

# Create a new FAISS vector store
alt_vector_store = FAISS.from_documents(
    documents=alt_movie_chunks,
    embedding=embedding_model
)

# Create top-3 retriever
alt_retriever = alt_vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("\nAlternative configuration created successfully!")
print("Chunk size: 1000")
print("Chunk overlap: 100")

Original number of chunks: 2044
Alternative number of chunks: 1111

Alternative configuration created successfully!
Chunk size: 1000
Chunk overlap: 100


In [ ]:
# Step 18: Rerun 2 questions using alternative chunking (1000/100)

def run_alt_rag_question(question):
    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    # Retrieve top-3 chunks using alternative retriever
    retrieved_docs = alt_retriever.invoke(question)

    print("\nTOP-3 RETRIEVED CHUNKS")
    print("-" * 100)

    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"\nChunk {i}")
        print("Movie:", doc.metadata.get("movie"))
        print("Wikipedia Title:", doc.metadata.get("title"))
        print("Content:")
        print(doc.page_content)
        print("\n" + "-" * 100)

    # Combine retrieved chunks into context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Format prompt
    formatted_prompt = rag_prompt.format(
        context=context,
        question=question
    )

    # Generate final answer
    answer = llm.invoke(formatted_prompt)

    print("\nFINAL RAG ANSWER:")
    print(answer)

    return retrieved_docs, answer


# Question 2: The Godfather
alt_q2_docs, alt_q2_answer = run_alt_rag_question(
    rag_questions[1]
)

# Question 5: Jurassic Park
alt_q5_docs, alt_q5_answer = run_alt_rag_question(
    rag_questions[4]
)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Who played Vito Corleone in The Godfather?

TOP-3 RETRIEVED CHUNKS
----------------------------------------------------------------------------------------------------

Chunk 1
Movie: The Godfather
Wikipedia Title: The Godfather
Content:
== Plot ==
In the mid-1940s, Michael Corleone is a World War II veteran and the youngest son of Don Vito Corleone. Vito is head of the Corleone mafia family and runs it with his eldest son Santino a.k.a. Sonny and adopted son Tom Hagen, a lawyer who is the family's consigliere. At his sister Connie's wedding to Carlo Rizzi, Michael introduces his family to his girlfriend, Kay Adams. Michael tells Kay stories of his family but makes it clear that he wants no part in his family's business.

----------------------------------------------------------------------------------------------------

Chunk 2
Movie: The Godfather
Wikipedia Title: The Godfather
Content:
Puzo was first to show interest in having Marlon Brando portray Don Vito Corleone by s

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL RAG ANSWER:
Marlon Brando played Vito Corleone in The Godfather.

QUESTION:
Who directed Jurassic Park?

TOP-3 RETRIEVED CHUNKS
----------------------------------------------------------------------------------------------------

Chunk 1
Movie: Jurassic Park (film)
Wikipedia Title: Jurassic Park
Content:
This article may be too long to read and navigate comfortably. Consider splitting content into sub-articles, condensing it, or adding subheadings. Please discuss this issue on the article's talk page. (July 2026)
Jurassic ParkTheatrical release poster by John AlvinDirected bySteven SpielbergScreenplay by
Michael Crichton
David Koepp
Based onJurassic Parkby Michael CrichtonProduced by
Kathleen Kennedy
Gerald R. Molen
Starring
Sam Neill
Laura Dern
Jeff Goldblum
Richard Attenborough
Bob Peck
Martin Ferrero
BD Wong
Samuel L. Jackson
Wayne Knight
Joseph Mazzello
Ariana Richards
CinematographyDean CundeyEdited byMichael KahnMusic byJohn WilliamsProductioncompanies
Universal Pictures[1

In [ ]:
# Step 19: Compare baseline and alternative chunking results

chunk_comparison = pd.DataFrame([
    {
        "Question": "Who played Vito Corleone in The Godfather?",
        "Baseline Chunk Size": 500,
        "Baseline Overlap": 50,
        "Baseline Result": "Incorrect",
        "Alternative Chunk Size": 1000,
        "Alternative Overlap": 100,
        "Alternative Result": "Correct",
        "Observation": (
            "Larger chunks provided more complete context linking "
            "Marlon Brando directly to Vito Corleone, and the LLM "
            "generated the correct answer."
        )
    },
    {
        "Question": "Who directed Jurassic Park?",
        "Baseline Chunk Size": 500,
        "Baseline Overlap": 50,
        "Baseline Result": "Incorrect",
        "Alternative Chunk Size": 1000,
        "Alternative Overlap": 100,
        "Alternative Result": "Correct",
        "Observation": (
            "With larger chunks, the first retrieved chunk contained "
            "the movie information and 'Directed by Steven Spielberg'. "
            "The final answer became correct, although the explanation "
            "included an inaccurate statement about Michael Crichton."
        )
    }
])

chunk_comparison

,Question,Baseline Chunk Size,Baseline Overlap,Baseline Result,Alternative Chunk Size,Alternative Overlap,Alternative Result,Observation
0,Who played Vito Corleone in The Godfather?,500,50,Incorrect,1000,100,Correct,Larger chunks provided more complete context l...
1,Who directed Jurassic Park?,500,50,Incorrect,1000,100,Correct,"With larger chunks, the first retrieved chunk ..."


### Q2 Summary
LangChain was used to implement a Retrieval-Augmented Generation (RAG) pipeline over ten movie Wikipedia pages. An initial chunk size of 500 characters with a 50-character overlap was used to divide the documents. The all-MiniLM-L6-v2 sentence-transformer model was used to embed the chunks, which were then saved in a FAISS vector database. A Qwen-based language model, an explicit prompt template, and a top-3 retriever were employed. Five movie-related questions that could be answered were assessed. The top three retrieved chunks contained the correct answer information for each of the five questions, resulting in a Retrieval Success Rate of 5/5 = 100%. Nevertheless, a correct final response was not always the outcome of successful retrieval. Two RAG generation errors were noted: When asked who portrayed Vito Corleone in The Godfather, the retrieved context included Marlon Brando, but the LLM gave Michael Corleone/Al Pacino as the incorrect response. The retrieved context explains the answer to the question about who directed Jurassic Park.


The chunk size and overlap for these two questions were altered from 500/50 to 1000/100 in order to examine the impact of chunking. Both questions yielded accurate final answers with the larger chunks. This implies that although they might also contain more unrelated information, larger chunks can offer the LLM more comprehensive contextual relationships. In the Jurassic Park experiment, a small hallucinated statement was included in the generated explanation even though the final answer was correct. Overall, the experiment demonstrates that generation quality and retrieval quality are distinct elements of a RAG system. The LLM may misinterpret or fail to use the evidence appropriately even if the right evidence is found.